In [ ]:
# Import the usual libraries and variables.
%run standard_imports.py

# Install a pip package in the current Jupyter kernel.
import sys
!{sys.executable} -m pip install --user pypika

In [ ]:
from pypika import MySQLQuery, Table, Order, Case, functions as fn

developer_app = Table("developer_app")
developer = Table("developer")

q = MySQLQuery.from_(developer_app).join(
    developer
).on(
    developer.id == developer_app.developer_id
).select(
#     developer.id,
#     developer.uuid,
#     developer.name,
    developer_app.id,
    developer_app.uuid,
    developer_app.name,
    developer_app.first_submitted_time
).where(
    developer.name != "Clover"
).where(
    (developer_app.approval_status == "PENDING") &
    (developer_app.deleted_time.isnull()) &
    (developer_app.first_submitted_time.notnull())
).orderby(developer_app.first_submitted_time, order=Order.asc)

print(q)
query = q.get_sql()

db = Db("~/.clover/p801.cfg")
df = pd.read_sql(query, con=db.conn)
db.close()

df

In [ ]:
import datetime

def days_since(x):
    current = datetime.datetime.utcnow()
    diff = current - x
    return diff.days
    
df["days_pending"] = df["first_submitted_time"].map(lambda x: days_since(x))
df

In [ ]:
# Plot length pending as a histogram.

plt.hist(df['days_pending'].dropna(), color='green', edgecolor='black', bins=int(df['days_pending'].max()/7))
plt.title('Histogram of Pending Apps')
plt.xlabel('Days Pending')
plt.ylabel('Apps')

print df.describe()

print "Range of days pending: " + str(df['days_pending'].min()) + ", " + str(df['days_pending'].max())
print "Mean (average) days pending: " + str(df['days_pending'].mean())
print "Median days pending: " + str(df['days_pending'].median())
print "Modal days pending: " + str(df['days_pending'].mode().values[0])

In [ ]:
# Get statuses of apps created in the past year.

db = Db("~/.clover/p801.cfg")
query = """
    SELECT
    developer_app.name,
    developer.name,
    TIMESTAMPDIFF(DAY, developer_app.created_time, curdate()),
    developer_app.approval_status AS 'developer_app.approval_status',
    developer_app.created_time,
    developer_app.modified_time
    FROM developer_app
    JOIN developer on developer.id = developer_app.developer_id
    JOIN account on account.id = developer.owner_account_id
    WHERE developer_app.name NOT LIKE "%disabled%"
    AND developer_app.name NOT LIKE "%test%"
    AND developer_app.name NOT LIKE "%experiment%"
    AND developer_app.name NOT LIKE "%demo%"
    AND""" + NOT_LIKE_DEVELOPER_NAMES_ACCOUNT_EMAILS + """
    AND account.last_login IS NOT NULL
    AND developer_app.created_time > DATE_SUB(curdate(), INTERVAL 1 YEAR)
    """
df = pd.read_sql(query, con=db.conn)
db.close()

df

In [ ]:
df['developer_app.approval_status'].value_counts()